In [1]:
!nvidia-smi

Wed Apr  8 18:01:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   30C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import numpy as np
import pandas as pd
import random
import os 
import torch
import polars as pl

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


# Load, Tiền xử lý 

In [4]:
# Đọc trực tiếp từ file JSONL theo chuẩn dữ liệu thô của Otto Kaggle
def load_and_flatten_jsonl(path):
    print(f"Loading {path}...")
    # Đọc nhanh jsonl, bung mảng events và tách các key thành cột riêng
    return (
        pl.read_ndjson(path)
          .explode("events")
          .unnest("events")
    )

# Bạn hãy điền lại chính xác đường dẫn file .jsonl trên Kaggle hiện tại của bạn
train_df = load_and_flatten_jsonl("/kaggle/input/datasets/organizations/otto/recsys-dataset/otto-recsys-train.jsonl")

# Bộ Test của Otto (Trong repo TRON tác giả dùng luôn test set làm validation during training)
test_df  = load_and_flatten_jsonl("/kaggle/input/datasets/organizations/otto/recsys-dataset/otto-recsys-test.jsonl")

print(train_df.head(2))
print(test_df.head(2))

Loading /kaggle/input/datasets/organizations/otto/recsys-dataset/otto-recsys-train.jsonl...
Loading /kaggle/input/datasets/organizations/otto/recsys-dataset/otto-recsys-test.jsonl...
shape: (2, 4)
┌─────────┬─────────┬───────────────┬────────┐
│ session ┆ aid     ┆ ts            ┆ type   │
│ ---     ┆ ---     ┆ ---           ┆ ---    │
│ i64     ┆ i64     ┆ i64           ┆ str    │
╞═════════╪═════════╪═══════════════╪════════╡
│ 0       ┆ 1517085 ┆ 1659304800025 ┆ clicks │
│ 0       ┆ 1563459 ┆ 1659304904511 ┆ clicks │
└─────────┴─────────┴───────────────┴────────┘
shape: (2, 4)
┌──────────┬────────┬───────────────┬────────┐
│ session  ┆ aid    ┆ ts            ┆ type   │
│ ---      ┆ ---    ┆ ---           ┆ ---    │
│ i64      ┆ i64    ┆ i64           ┆ str    │
╞══════════╪════════╪═══════════════╪════════╡
│ 12899779 ┆ 59625  ┆ 1661724000278 ┆ clicks │
│ 12899779 ┆ 875854 ┆ 1661724026702 ┆ clicks │
└──────────┴────────┴───────────────┴────────┘


In [5]:
print(f"train sessions: {train_df['session'].n_unique()}")
print(f"test  sessions: {test_df['session'].n_unique()}")

train sessions: 12899779
test  sessions: 1671803


In [6]:
# single object nên là lọc clicks, đổi tên cột session =>session_id
def filter_clicks(df):
    return (
        df.filter(pl.col("type") == "clicks")
          .rename({"session": "session_id"})
          .sort(["session_id", "ts"])
    )
    
train_df = filter_clicks(train_df)
test_df  = filter_clicks(test_df)

print(f"train sessions: {train_df['session_id'].n_unique()}")
print(f"test  sessions: {test_df['session_id'].n_unique()}")

train sessions: 12899779
test  sessions: 1671803


In [7]:
# Otto dataset đã được encode sẵn dưới dạng chuỗi các integer (0-1855602)
# Code TRON gốc (preprocessing.py - run_preprocessing_otto) CHỈ cần tăng ID lên 1 (không cần lọc support, lọc độ dài hay map lại idx)
# Điều này để nhường ID 0 cho làm phần padding.

train_df = train_df.with_columns(pl.col("aid") + 1)
test_df  = test_df.with_columns(pl.col("aid") + 1)

num_items = train_df["aid"].max() # Khớp với max ID sau khi đã + 1
print(f"num_items  : {num_items}")
print(f"train sessions: {train_df['session_id'].n_unique()}")
print(f"test  sessions: {test_df['session_id'].n_unique()}")
print(train_df.head(5))

num_items  : 1855603
train sessions: 12899779
test  sessions: 1671803
shape: (5, 4)
┌────────────┬─────────┬───────────────┬────────┐
│ session_id ┆ aid     ┆ ts            ┆ type   │
│ ---        ┆ ---     ┆ ---           ┆ ---    │
│ i64        ┆ i64     ┆ i64           ┆ str    │
╞════════════╪═════════╪═══════════════╪════════╡
│ 0          ┆ 1517086 ┆ 1659304800025 ┆ clicks │
│ 0          ┆ 1563460 ┆ 1659304904511 ┆ clicks │
│ 0          ┆ 1309447 ┆ 1659367439426 ┆ clicks │
│ 0          ┆ 16247   ┆ 1659367719997 ┆ clicks │
│ 0          ┆ 1781823 ┆ 1659367871344 ┆ clicks │
└────────────┴─────────┴───────────────┴────────┘


# Config

In [8]:
config = {
   "model": "sasrec",
    "dataset": "otto",
    "hidden_size": 200,
    "num_layers": 2,
    "dropout": 0.05,
    "num_batch_negatives": 127,
    "num_uniform_negatives": 8192,
    "reject_uniform_session_items": False,
    "reject_in_batch_items": True ,
    "sampling_style": "sessionwise",
    "loss": "ssm",
    "max_epochs": 1,
    "batch_size": 128,
    "max_session_length": 50,
    "lr": 0.0005,
    "limit_val_batches": 1.0,
    "accelerator": "gpu",
    "overfit_batches": 0,
    "share_embeddings": True,
    "output_bias": False,
    "shuffling_style": "no_shuffling",
    "optimizer": "adam"
}

# shared.logits_computation

In [9]:
from torch import concat

def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))


def lookup_and_multiply(prediction_head, positives, uniform_negatives, in_batch_negatives, embedding_layer, sampling_style):
    positive_logits = multiply_head_with_embedding(prediction_head.unsqueeze(-2),
                                                   embedding_layer(positives).unsqueeze(-2)).squeeze(-1)

    if sampling_style == "eventwise":
        uniform_negative_logits = multiply_head_with_embedding(prediction_head.unsqueeze(-2),
                                                               embedding_layer(uniform_negatives)).squeeze(-2)
    else:
        uniform_negative_logits = multiply_head_with_embedding(prediction_head, embedding_layer(uniform_negatives))

    in_batch_negative_logits = multiply_head_with_embedding(prediction_head, embedding_layer(in_batch_negatives))
    negative_logits = concat([uniform_negative_logits, in_batch_negative_logits], dim=-1)
    return positive_logits, negative_logits

# shared.loss

In [10]:
import torch
from torch import cat, exp, log, sigmoid, softmax, sum, tensor
from torch.nn import CrossEntropyLoss

ce_loss = CrossEntropyLoss(reduction="none")


def _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target):
    sm_logits = cat((pos_logits, neg_logits), dim=-1)
    shape = sm_logits.shape
    return ce_loss(sm_logits.reshape([-1, shape[-1]]), target).reshape([shape[0], shape[1]]) * mask


def sampled_softmax_loss(pos_logits, neg_logits, mask, device="cpu"):
    target = tensor([0], device=device).tile(mask.numel())
    elementwise_ssm_loss = _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target)
    return sum(elementwise_ssm_loss) / sum(mask)


def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = log(1. + exp(-pos_logits) + epsilon) + log(1. + exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()


def _diff_logits(pos_logits, neg_logits):
    return (pos_logits - neg_logits)


def _elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits):
    logits_diff = sigmoid(_diff_logits(pos_logits, neg_logits))
    s_j = softmax(neg_logits - torch.max(neg_logits, dim=-1)[0].unsqueeze(-1), dim=-1)
    return s_j * logits_diff


def _bpr_max_loss_unregulized(pos_logits, neg_logits, mask):
    bpr_max_loss_per_element = -log(sum(_elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits), dim=-1))
    return bpr_max_loss_per_element, sum(bpr_max_loss_per_element * mask) / sum(mask)


def _bpr_max_loss_regularization(neg_logits, penalty, mask):
    regularization = penalty * sum(softmax(neg_logits, dim=-1) * neg_logits * neg_logits, dim=-1)
    return sum(regularization * mask) / sum(mask)


def bpr_max_loss(penalty, pos_logits, neg_logits, mask, device="cpu"):
    _, unregulized_bpr_max_loss = _bpr_max_loss_unregulized(pos_logits, neg_logits, mask)
    return unregulized_bpr_max_loss + _bpr_max_loss_regularization(neg_logits, penalty, mask)


def calc_loss(loss_fn,
              x_hat,
              labels,
              uniform_negatives,
              in_batch_negatives,
              mask,
              embeddings,
              sampling_style,
              final_activation,
              topk_sampling=False,
              topk_sampling_k=1000,
              device="cpu"):
    pos_logits, neg_logits = lookup_and_multiply(x_hat, labels, uniform_negatives, in_batch_negatives, embeddings,
                                                 sampling_style)
    if topk_sampling:
        neg_logits, _ = torch.topk(neg_logits, k=topk_sampling_k, dim=-1)
    pos_scores, neg_scores = final_activation(pos_logits), final_activation(neg_logits)
    return loss_fn(pos_scores, neg_scores, mask, device=device)


# shared.sample

In [11]:
import itertools
from random import sample

import numpy as np

def _uniform_negatives(num_items, shape):
    return np.random.randint(1, num_items+1, shape)

def _uniform_negatives_session_rejected(num_items, shape, in_session_items):
        negatives = []
        for _ in range(np.prod(shape)):
            negative = np.random.randint(1, num_items+1)
            while negative in in_session_items:
                negative = np.random.randint(1, num_items+1)
            negatives.append(negative)
        return np.array(negatives).reshape(shape)

def _infer_shape(session_len, num_uniform_negatives, sampling_style):
    if sampling_style=="eventwise":
        return [session_len, num_uniform_negatives]
    elif sampling_style=="sessionwise":
        return [num_uniform_negatives]
    else:
         return []

def sample_uniform(num_items, shape, in_session_items, reject_session_items):
    if reject_session_items:
        return _uniform_negatives_session_rejected(num_items, shape, in_session_items)
    else: 
        return _uniform_negatives(num_items, shape)

def sample_uniform_negatives_with_shape(clicks, num_items, session_len, num_uniform_negatives, sampling_style, reject_session_items):
    in_session_items = set(clicks)
    shape = _infer_shape(session_len, num_uniform_negatives, sampling_style)
    if shape:
        negatives = sample_uniform(num_items, shape, in_session_items, reject_session_items)
    else: 
        negatives = np.array([])
    return negatives


def sample_in_batch_negatives(batch_positives, num_in_batch_negatives, batch_session_len, reject_session_items):
    in_batch_negatives = []
    positive_indices = itertools.accumulate(batch_session_len)
    positive_indices = [0] + [p for p in positive_indices]
    if reject_session_items:
        for i in range(len(positive_indices[:-1])):
            candidate_positives = batch_positives[:positive_indices[i]] + batch_positives[
                positive_indices[i + 1]:]
            in_batch_negatives.append(sample(candidate_positives, num_in_batch_negatives))
    else:
        for i in range(len(batch_session_len)):
            in_batch_negatives.append(sample(batch_positives, num_in_batch_negatives))
    return in_batch_negatives

# shared.utils

In [12]:
# def get_offsets(sessions_path):
#         line_offsets = []
#         with open(sessions_path, "rt") as f:
#             offset = 0
#             for line_idx, line in enumerate(f):
#                 line_len = len(line)
#                 line_offsets.append((line_len, line_idx, offset))
#                 offset += line_len
#         line_offsets = [offset for _, _, offset in line_offsets]
#         return line_offsets



# shared.evaluate

In [13]:
from torch import cumsum, flip, inf, max, stack, sum, topk, where

def calculate_ranks(logits, labels, cutoffs):
    num_logits = logits.shape[-1]
    k = min(num_logits, max(cutoffs).item())
    _, indices = topk(logits, k=k, dim=-1)
    indices = flip(indices, dims=[-1])
    hits = indices == labels.unsqueeze(dim=-1)
    ranks = sum(cumsum(hits, -1), -1) - 1.
    ranks[ranks == -1] = float('inf')
    return ranks


def pointwise_mrr(ranks, cutoffs, mask):
    res = where(ranks < cutoffs.unsqueeze(-1).unsqueeze(-1), ranks, float('inf'))
    return (1 / (res + 1)) * mask


def pointwise_recall(ranks, cutoffs, mask):
    res = ranks < cutoffs.unsqueeze(-1).unsqueeze(-1)
    return res.float() * mask


def mean_metric(pointwise_metric, mask):
    hits = sum(pointwise_metric, dim=(2, 1))
    return hits / sum(mask).clamp(0.0000005)


def validate_batch_per_timestamp(batch, x_hat, output_embedding, cut_offs):
    recalls = []
    mrrs = []
    for t in range(x_hat.shape[1]):
        mask = batch['mask'][:, t]
        positives = batch['labels'][:, t]
        logits = multiply_head_with_embedding(x_hat[:, t], output_embedding.weight)
        logits[:, 0] = -inf  # set score for padding item to -inf
        ranks = calculate_ranks(logits, positives, cut_offs)
        pw_rec = pointwise_recall(ranks, cut_offs, mask)
        recalls.append(pw_rec.squeeze(dim=1))
        pw_mrr = pointwise_mrr(ranks, cut_offs, mask)
        mrrs.append(pw_mrr.squeeze(dim=1))
    pw_rec = stack(recalls, dim=2)
    pw_mrr = stack(mrrs, dim=2)
    return mean_metric(pw_rec, batch["mask"]), mean_metric(pw_mrr, batch["mask"])


# sasrec.dataset

In [14]:
import json

import numpy as np
import torch
from torch.utils.data.dataset import Dataset
class SasRecDataset(Dataset):

    def __init__(self,
                 df, #sửa thành df thay vì path vì trong model gốc đọc từ file jsonl
                 num_items,
                 max_seqlen,
                 num_uniform_negatives=1,
                 num_in_batch_negatives=0,
                 reject_uniform_session_items=False,
                 reject_in_batch_items=True,
                 sampling_style="eventwise",
                 shuffling_style="no_shuffling"
                 ):
        # self.session_path = sessions_path
        # self.total_sessions = total_sessions
        self.num_items = num_items
        self.max_seqlen = max_seqlen
        self.shuffling_style = shuffling_style
        self.num_uniform_negatives = num_uniform_negatives
        self.num_in_batch_negatives = num_in_batch_negatives
        self.reject_uniform_session_items = reject_uniform_session_items
        self.reject_in_batch_items = reject_in_batch_items
        self.sampling_style = sampling_style

        assert self.sampling_style in {"eventwise", "sessionwise", "batchwise"}
        # assert len(self.line_offsets) == self.total_sessions, f"{len(self.line_offsets)} != {self.total_sessions}"
        self.session_list = (
            df.sort(["session_id", "ts"])
              .group_by("session_id")
              .agg(pl.col("aid").sort_by("ts").alias("aids"))
              ["aids"]
              .to_list()
        )
        print(f"Done: {len(self.session_list)} sessions loaded")

    def __len__(self):
        return len(self.session_list)

    def __getitem__(self, idx):


        if self.shuffling_style=="shuffle_with_replacement":
            idx = np.random.randint(0,len(self.session_list))

        # f.seek(self.line_offsets[idx])
        # line = f.readline()
        # session = json.loads(line)
        # session = session["events"]

        # assert sorted(session, key=lambda d: d["ts"]) == session

        # clicks = [int(event["aid"]) for event in session if event["type"] == "clicks"]
        clicks = [int(c) for c in self.session_list[idx]]
        clicks = clicks[-(self.max_seqlen + 1):]
        session_len = min(len(clicks) - 1, self.max_seqlen)
        labels = clicks[1:]
        clicks = clicks[:-1]
        negatives = sample_uniform_negatives_with_shape(clicks, self.num_items, session_len, self.num_uniform_negatives, self.sampling_style, self.reject_uniform_session_items)

        return {'clicks': clicks, 'labels': labels, 'session_len': session_len, "uniform_negatives": negatives.tolist()}


    def dynamic_collate(self, batch):
        batch_clicks = list()
        batch_mask = list()
        batch_labels = list()
        batch_session_len = list()
        batch_positives = list()
        max_len = self.max_seqlen
        batch_uniform_negatives = list()
        in_batch_negatives = list()

        for item in batch:
            session_len = item["session_len"]
            batch_clicks.append((max_len - session_len) * [0] + item["clicks"])
            batch_mask.append((max_len - session_len) * [0.] + session_len * [1.])
            batch_labels.append((max_len - session_len) * [0] + item["labels"])
            batch_session_len.append(session_len)
            batch_positives.extend(item["clicks"])

            if self.sampling_style=="eventwise":
                batch_uniform_negatives.append((max_len - session_len) * [[0]*self.num_uniform_negatives] + item["uniform_negatives"]) 
            elif self.sampling_style=="sessionwise":
                batch_uniform_negatives.append(item["uniform_negatives"]) 
            
        if self.sampling_style=="batchwise":
            batch_uniform_negatives = sample_uniform(self.num_items, [self.num_uniform_negatives], set(batch_positives), self.reject_in_batch_items) 

        in_batch_negatives = sample_in_batch_negatives(batch_positives, self.num_in_batch_negatives, batch_session_len, self.reject_in_batch_items) 
        
        return {
            'clicks': torch.tensor(batch_clicks, dtype=torch.long),
            'labels': torch.tensor(batch_labels, dtype=torch.long), 
            'mask': torch.tensor(batch_mask, dtype=torch.float),
            'session_len': torch.tensor(batch_session_len, dtype=torch.long),
            'in_batch_negatives': torch.tensor(in_batch_negatives, dtype=torch.long), 
            'uniform_negatives': torch.tensor(batch_uniform_negatives, dtype=torch.long) 
        }

# sasrec.model

In [15]:
from functools import partial
from pathlib import Path

import numpy as np
import pytorch_lightning as py_light
import torch
from torch import concat, diag, logical_and, logical_or, nn, tensor, tile
from torch.nn import Dropout

class DynamicPositionEmbedding(torch.nn.Module):

    def __init__(self, max_len, dimension):
        super(DynamicPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.pos_indices = torch.arange(0, self.max_len, dtype=torch.int)
        self.register_buffer('pos_indices_const', self.pos_indices)

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x


class SASRec(py_light.LightningModule):

    def __init__(self,
                 hidden_size,
                 dropout_rate,
                 max_len,
                 num_items,
                 batch_size,
                 sampling_style,
                 topk_sampling=False,
                 topk_sampling_k=1000,
                 learning_rate=0.001,
                 num_layers=2,
                 loss='bce',
                 bpr_penalty=None,
                 optimizer='adam',
                 output_bias=False,
                 share_embeddings=True,
                 final_activation=False):
        super(SASRec, self).__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.future_mask = torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1)
        self.register_buffer('future_mask_const', self.future_mask)
        self.register_buffer('seq_diag_const', ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones', torch.ones([self.batch_size, self.max_len, 1]))
        if output_bias and share_embeddings:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)

        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)

        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif (not share_embeddings) and output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)

        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size,
                                                   nhead=1,
                                                   dim_feedforward=hidden_size,
                                                   dropout=dropout_rate,
                                                   batch_first=True,
                                                   norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)
        self.merge_attn_mask = True
        if final_activation:
            self.final_activation = nn.ELU(0.5)
        else:
            self.final_activation = nn.Identity()

        self.loss_fn = loss
        if self.loss_fn == 'bce':
            self.loss = bce_loss
        elif self.loss_fn == 'ssm':
            self.loss = sampled_softmax_loss
        elif self.loss_fn == 'bpr-max':
            if bpr_penalty is not None:
                self.loss = partial(bpr_max_loss, bpr_penalty)
            else:
                raise ValueError('bpr_penalty must be provided for bpr_max loss')
        else:
            raise ValueError('Loss function not supported')

        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]

        if not self.merge_attn_mask:
            return self.future_mask_const[:seq_len, :seq_len]

        padding_mask_broadcast = ~padding_mask.bool().unsqueeze(1)
        future_masks = tile(self.future_mask_const[:seq_len, :seq_len], (batch_size, 1, 1))
        merged_masks = logical_or(padding_mask_broadcast, future_masks)
        # Always allow self-attention to prevent NaN loss
        # See: https://github.com/pytorch/pytorch/issues/41508
        diag_masks = tile(self.seq_diag_const[:seq_len, :seq_len], (batch_size, 1, 1))
        return logical_and(diag_masks, merged_masks)

    def forward(self, item_indices, mask):
        att_mask = self.merge_attn_masks(mask)
        items = self.item_embedding(
            item_indices)[:, :, :-1] if self.output_bias and self.share_embeddings else self.item_embedding(item_indices)
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones], dim=-1) if self.output_bias else x

    def training_step(self, batch, _):
        x_hat = self.forward(batch["clicks"], batch["mask"])
        train_loss = calc_loss(self.loss, x_hat, batch["labels"], batch["uniform_negatives"], batch["in_batch_negatives"],
                               batch["mask"], self.output_embedding, self.sampling_style, self.final_activation,
                               self.topk_sampling, self.topk_sampling_k, self.device)
        self.log("train_loss", train_loss)
        return train_loss

    def validation_step(self, batch, _batch_idx):
        x_hat = self.forward(batch['clicks'], batch['mask'])
        cut_offs = tensor([5, 10, 20], device=self.device)
        recall, mrr = validate_batch_per_timestamp(batch, x_hat, self.output_embedding, cut_offs)
        test_loss = calc_loss(self.loss, x_hat, batch["labels"], batch["uniform_negatives"], batch["in_batch_negatives"],
                              batch["mask"], self.output_embedding, self.sampling_style, self.final_activation,
                              self.topk_sampling, self.topk_sampling_k, self.device)
        for i, k in enumerate(cut_offs.tolist()):
            self.log(f'recall_cutoff_{k}', recall[i])
            self.log(f'mrr_cutoff_{k}', mrr[i])
        self.log('test_seq_len', x_hat.shape[1])
        self.log('test_loss', test_loss)

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            optimizer = torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        else:
            raise ValueError('Optimizer not supported, please use adam or adagrad')
        return optimizer

    def export_topk_onnx(self, out_dir):
        top_k_model = TopKModel(self)
        top_k_model.export_onnx(out_dir)

    def export(self, out_dir):
        self.export_topk_onnx(out_dir)


class TopKModel(py_light.LightningModule):

    def __init__(self, model: SASRec):
        super(TopKModel, self).__init__()
        self.model = model
        # example input for self.forward(item_indices, k)
        self.example_input_array = (torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]), torch.tensor(10))

    def forward(self, item_indices, k):
        mask = torch.ones(item_indices.shape[0]).unsqueeze(0)
        self.model.merge_attn_mask = False
        x_hat = self.model.forward(item_indices.unsqueeze(0), mask)[:, -1]
        logits = multiply_head_with_embedding(x_hat, self.model.item_embedding.weight)
        logits[0][0] = -torch.inf  # set score for padding item to -inf
        scores, indices = torch.topk(logits, k)
        return indices.squeeze(0), scores.squeeze(0)

    def export_onnx(self, out_dir, verbose=True):
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        self.to_onnx(f"{out_dir}/sasrec.onnx",
                     export_params=True,
                     opset_version=14,
                     verbose=verbose,
                     do_constant_folding=False,
                     input_names=["item_indices", "k"],
                     output_names=[f"indices", "scores"],
                     dynamic_axes={
                         'item_indices': {
                             0: 'sequence'
                         },
                         'indices': {
                             0: 'k'
                         },
                         'scores': {
                             0: 'k'
                         }
                     })

# sasrec.train

In [16]:
import os
from pytorch_lightning.trainer.trainer import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar

def train_sasrec(config, train_df, test_df, num_items):
    checkpoint_callback = ModelCheckpoint(save_top_k=1,
                                          monitor='recall_cutoff_20',
                                          mode='max',
                                          filename=f'sasrec-{config["dataset"]}-' + '{epoch}-{recall_cutoff_20:.3f}')
    
    trainer = Trainer(max_epochs=config["max_epochs"],
                      precision=16,
                      limit_val_batches=config["limit_val_batches"],
                      log_every_n_steps=1,
                      accelerator=config["accelerator"],
                      devices=1,
                      overfit_batches=config["overfit_batches"],
                      callbacks=[checkpoint_callback])
    
    assert 0 <= config["num_batch_negatives"] < config['batch_size']

    print("Creating train dataset...")
    train_set = SasRecDataset(df = train_df,
                              num_items=num_items,
                              max_seqlen=config["max_session_length"],
                              num_in_batch_negatives=config["num_batch_negatives"],
                              num_uniform_negatives=config["num_uniform_negatives"],
                              reject_uniform_session_items=config["reject_uniform_session_items"],
                              reject_in_batch_items=config["reject_in_batch_items"],
                              sampling_style=config["sampling_style"],
                              shuffling_style=config["shuffling_style"])
    print("Creating valid dataset (from test_df)...")
    test_set = SasRecDataset(df = test_df,
                             num_items=num_items,
                             max_seqlen=config["max_session_length"],
                             num_in_batch_negatives=config["num_batch_negatives"],
                             num_uniform_negatives=config["num_uniform_negatives"],
                             reject_uniform_session_items=config["reject_uniform_session_items"],
                             reject_in_batch_items=config["reject_in_batch_items"],
                             sampling_style=config["sampling_style"],
                             shuffling_style="no_shuffling")

    shuffle = True if config["shuffling_style"] == "shuffle_without_replacement" else False
    train_loader = DataLoader(train_set,
                              drop_last=True,
                              batch_size=config["batch_size"],
                              shuffle=shuffle,
                              pin_memory=True,
                              persistent_workers=True,
                              num_workers=4, # Set to 4 (safe for kaggle), thay vì os.cpu_count() dễ gây Killed process
                              collate_fn=train_set.dynamic_collate)

    test_loader = DataLoader(test_set,
                             drop_last=True,
                             batch_size=config["batch_size"],
                             shuffle=False,
                             pin_memory=True,
                             persistent_workers=True,
                             num_workers=4, # Set to 4 (safe for kaggle)
                             collate_fn=test_set.dynamic_collate)

    model = SASRec(hidden_size=config["hidden_size"],
                   dropout_rate=config["dropout"],
                   max_len=config["max_session_length"],
                   num_items=num_items,
                   batch_size=config["batch_size"],
                   sampling_style=config["sampling_style"],
                   topk_sampling=config.get("topk_sampling", False),
                   topk_sampling_k=config.get("topk_sampling_k", 1000),
                   learning_rate=config["lr"],
                   num_layers=config["num_layers"],
                   loss=config["loss"],
                   bpr_penalty=config.get("bpr_penalty", None),
                   optimizer=config["optimizer"],
                   output_bias=config["output_bias"],
                   share_embeddings=config["share_embeddings"])

    return trainer, model, train_loader, test_loader

In [17]:
trainer, model, train_loader, test_loader = train_sasrec(
    config, train_df, test_df, num_items
)
trainer.fit(model, train_loader, val_dataloaders=test_loader)

# --- PHẦN LƯU MODEL DÀNH CHO KAGGLE OUTPUT ---
import shutil

print("Training finished! Saving model and checkpoints...")

# 1. Pytorch Lightning đã tự động lưu file Checkpoint (best model) qua ModelCheckpoint callback.
# Thường nằm trong thư mục thư mục 'lightning_logs/version_0/checkpoints/...'
# Chúng ta sẽ copy checkpoint tốt nhất ra thư mục làm việc chính (/kaggle/working) để dễ download.
best_model_path = trainer.checkpoint_callback.best_model_path
if best_model_path:
    print(f"Best model found at: {best_model_path}")
    shutil.copy(best_model_path, "./best_sasrec_model.ckpt")
    print("Saved best checkpoint to: ./best_sasrec_model.ckpt")

# 2. Lưu model dưới định dạng ONNX (rất nhẹ, dùng để load chạy Inference siêu nhanh)
# Hàm export_onnx đã được tác giả code sẵn bên trong class TopKModel.
print("Exporting ONNX model for fast inference...")
model.export("./onnx_export")
print("Saved ONNX model to: ./onnx_export/sasrec.onnx")

print("All done!")

/usr/local/lib/python3.12/dist-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..


Creating train dataset...
Done: 12899779 sessions loaded
Creating valid dataset (from test_df)...
Done: 1671803 sessions loaded


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA H100 80GB HBM3') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matm

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                       ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ item_embedding             │ Embedding                │  371 M │ train │     0 │
│ 1 │ positional_embedding_layer │ DynamicPositionEmbedding │ 10.0 K │ train │     0 │
│ 2 │ norm                       │ LayerNorm                │    400 │ train │     0 │
│ 3 │ input_dropout              │ Dropout                  │      0 │ train │     0 │
│ 4 │ encoder                    │ TransformerEncoder       │  484 K │ train │     0 │
│ 5 │ final_activation           │ Identity                 │      0 │ train │     0 │
└───┴────────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 371 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 371 M                                                                                                
Total estimated model params size (MB): 1.5 K                                                                      
Modules in train mode: 28                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()